In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam

# ---------------------------------------------------------
# PATHS
# ---------------------------------------------------------
DATASET_PATH = "/content/drive/MyDrive/Datasets/banana"
MODEL_SAVE_PATH = "/content/drive/MyDrive/Datasets/banana/model_banana_disease.keras"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10


# ---------------------------------------------------------
# Load dataset
# ---------------------------------------------------------
def load_dataset(path):
    filepaths = []
    labels = []

    for class_name in os.listdir(path):
        class_path = os.path.join(path, class_name)

        if os.path.isdir(class_path):
            for img in os.listdir(class_path):
                if img.lower().endswith((".jpg", ".png", ".jpeg")):
                    filepaths.append(os.path.join(class_path, img))
                    labels.append(class_name)

    return pd.DataFrame({"Filepath": filepaths, "Label": labels})


# ---------------------------------------------------------
# Hybrid Model
# ---------------------------------------------------------
def build_model(num_classes):

    effnet = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(224, 224, 3)
    )
    effnet.trainable = False

    convnext = tf.keras.applications.ConvNeXtTiny(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(224, 224, 3)
    )
    convnext.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))

    x1 = effnet(inputs)
    x2 = convnext(inputs)

    x = tf.keras.layers.Concatenate()([x1, x2])
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs)


# ---------------------------------------------------------
# Train
# ---------------------------------------------------------
def train():

    print("GPU Available:", tf.config.list_physical_devices('GPU'))

    print("Loading dataset...")
    df = load_dataset(DATASET_PATH)

    # 🔥 Manual split
    train_df, val_df = train_test_split(
        df,
        train_size=0.8,
        stratify=df["Label"],
        random_state=42
    )

    # 🔥 Strong augmentation
    train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
        rotation_range=35,
        zoom_range=0.4,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.3,
        brightness_range=[0.5, 1.5],
        horizontal_flip=True,
        fill_mode="nearest"
    )

    val_gen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
    )

    train_images = train_gen.flow_from_dataframe(
        train_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    val_images = val_gen.flow_from_dataframe(
        val_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    print("Classes:", train_images.class_indices)

    model = build_model(len(train_images.class_indices))

    model.compile(
        optimizer=Adam(1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    print("Training on GPU 🚀 ...")
    history = model.fit(
        train_images,
        validation_data=val_images,
        epochs=EPOCHS
    )

    print("Saving model...")
    model.save(MODEL_SAVE_PATH)

    # Save plots
    plt.figure()
    plt.plot(history.history["accuracy"])
    plt.plot(history.history["val_accuracy"])
    plt.legend(["Train", "Val"])
    plt.title("Accuracy")
    plt.savefig("/content/drive/MyDrive/Datasets/banana/accuracy.png")
    plt.close()

    plt.figure()
    plt.plot(history.history["loss"])
    plt.plot(history.history["val_loss"])
    plt.legend(["Train", "Val"])
    plt.title("Loss")
    plt.savefig("/content/drive/MyDrive/Datasets/banana/loss.png")
    plt.close()


train()

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Loading dataset...
Found 3740 validated image filenames belonging to 4 classes.
Found 935 validated image filenames belonging to 4 classes.
Classes: {'Cordana': 0, 'Healthy': 1, 'Panama Disease': 2, 'Yellow and Black Sigatoka': 3}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training on GPU 🚀 ...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 1652s 14s/step - accuracy: 0.5480 - loss: 1.0649 - val_accuracy: 0.8545 - val_loss: 0.4129
Epoch 2/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 92s 787ms/step - accuracy: 0.7995 - loss: 0.5272 - val_accuracy: 0.9155 - val_loss: 0.2461
Epoch 3/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 91s 775ms/step - accuracy: 0.8884 - loss: 0.3294 - val_accuracy: 0.9433 - val_loss: 0.1669
Epoch 4/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 91s 773ms/step - accuracy: 0.9172 - loss: 0.2464 - val_accuracy: 0.9540 - val_loss: 0.1339
Epoch 5/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 90s 769ms/step - accuracy: 0.9219 - loss: 0.2171 - val_accuracy: 0.9636 - val_loss: 0.1176
Epoch 6/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 95s 810ms/step - accuracy: 0.9350 - loss: 0.1838 - val_accuracy: 0.9604 - val_loss: 0.1069
Epoch 7/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 92s 784ms/step - accuracy: 0.9432 - loss: 0.1603 - val_accuracy: 0.9679 - val_loss: 0.0955
Epoch 8/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 144s 800ms/step - accuracy: 0.9422 - loss: 